# Automated Land Cover Sampling Strategy
### This notebook performs an end-to-end spatial analysis to generate high-quality training samples from Corine Land Cover (CLC) data and Sentinel-2 NDVI imagery, for the year 2018.
This is the V5 (Fixed Distribution): Approximately 200 polygons sampled across all 28 classes including 4 seasons. 

### inputs : 
1. one raster of NDVI, 
2. one cropped shp of CLC or any local reference layer
### outputs: 
1. Square polygon of of pure samples 
2. Randomly selected up to 60 polygons per class

## Step 0: Environment Setup

In [20]:
# 1. Import libraries
import geopandas as gpd
import pandas as pd
import rasterio
from rasterstats import zonal_stats
import folium
from folium import plugins
import json
import ee
import os
from ipyleaflet import Map, DrawControl
import time
import datetime
import math, requests
from pathlib import Path
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io
import ipywidgets as widgets
from IPython.display import display, Javascript, HTML, clear_output, Markdown
import numpy as np
from rasterio.mask import mask
from shapely.geometry import Point, box, mapping
from shapely.ops import unary_union, transform
from shapely.validation import make_valid
from sklearn.ensemble import IsolationForest
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import textwrap
import pyproj
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
import textwrap
import random
from cdsetool.query import query_features
from cdsetool.download import download_features
from cdsetool.credentials import Credentials
import zipfile
from rasterio.warp import reproject, Resampling
from rasterio.crs import CRS
import shutil

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)

print ("Libraries imported correctly!")

Libraries imported correctly!


In [2]:
# 2. Initialize Earth Engine and Select bounding box AOI

# Initialize (authenticate once with ee.Authenticate() if needed)
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print ("Earth Engine initialized!")

SEASONS = {
    "winter": ("2024-12-01", "2025-02-28"),
    "spring": ("2025-03-01", "2025-05-31"),
    "summer": ("2025-06-01", "2025-08-31"),
    "autumn": ("2025-09-01", "2025-11-30")
}

def mask_s2_clouds(image):

    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(
            qa.bitwiseAnd(cirrus_bit_mask).eq(0)
        )
    )

    return (
        image.updateMask(mask)
        .divide(10000)
    )

def get_seasonal_ndvi(aoi, start_date, end_date):

    s2 = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                20
            )
        )
        .map(mask_s2_clouds)
    )

    composite = s2.median()

    ndvi = (
        composite.normalizedDifference(
            ["B8", "B4"]
        )
        .rename("NDVI")
    )

    return ndvi

# =========================
# AOI SELECTION 
# =========================

mode = input("Select area input mode ('bbox' or 'map'): ").strip().lower()

bbox = None  # <- unified variable

if mode == "map":
    
    bbox_area = {"bounds": None}
    
    m = Map(center=(52, 4), zoom=4)
    
    draw_control = DrawControl(
        rectangle={"shapeOptions": {"fillOpacity": 0.1}},
        polygon={},
        polyline={},
        circle={},
        circlemarker={},
        marker={}
    )
    
    def handle_draw(target, action, geo_json):
        if action == "created":
            coords = geo_json["geometry"]["coordinates"][0]
    
            lons = [c[0] for c in coords]
            lats = [c[1] for c in coords]
    
            bbox_area["bounds"] = [
                min(lons),  # west
                min(lats),  # south
                max(lons),  # east
                max(lats)   # north
            ]
    
            print("✅ Bounding box stored:", bbox_area["bounds"])
    
    draw_control.on_draw(handle_draw)
    m.add_control(draw_control)
    
    display(m)
    
    print("➡️ Draw rectangle, then run next cell and assign bbox = bbox_area['bounds']")

else:
    user_input = input("Enter AOI bbox [west, south, east, north] (like - [-6.35, 36.5, -5.95, 36.9]):  ")
    
    try:
        bbox = eval(user_input)
        if not (isinstance(bbox, list) and len(bbox) == 4):
            raise ValueError
        print(f"✅ Coordinates set to: {bbox}")
    except Exception:
        raise ValueError("Invalid format. Use: [-6.35, 36.5, -5.95, 36.9]")

# ✅ Create GEE AOI (still needed for other parts of your code)
if bbox is not None:
    aoi = ee.Geometry.Rectangle(bbox)

Earth Engine initialized!


Select area input mode ('bbox' or 'map'):  bbox
Enter AOI bbox [west, south, east, north] (like - [-6.35, 36.5, -5.95, 36.9]):   [-6.35, 36.5, -5.95, 36.9]


✅ Coordinates set to: [-6.35, 36.5, -5.95, 36.9]


In [3]:
# 3. Setup Paths
root_dir = r'P:\Nature\Biodiversity\EcosystemAcc\CopernicusECOACC\TrainingSamples\Spain\Model_v2\CLC_sampling_process_SPAIN'
input_shp = os.path.join(root_dir, 'CLC_L3_2018_spain.shp')
raster_path = os.path.join(root_dir, 'NDVI_Tile_29SQB_2018_spain.tif')
SEASON_OUTPUT = os.path.join(
    root_dir,
    "seasonal_outputs"
)

# Define sub-folders
folders = ['split_outputs', 'processed_poly', 'merged_reclassified', 'exports']
paths = {folder: os.path.join(root_dir, folder) for folder in folders}

# Create all necessary directories
for path in paths.values():
    if not os.path.exists(path):
        os.makedirs(path)

print("Libraries imported and directories initialized.")

Libraries imported and directories initialized.


## Step 1: Reclassification and Class Splitting

We simplify the CLC Level 3 codes into broader categories (Level 2) where necessary, then split the dataset into individual files based on the class.

In [4]:
# 4. Change the values for reclassification based on the need
def log_time(msg, start=None):
    now = time.time()
    ts = datetime.now().strftime("%H:%M:%S")
    if start:
        print(f"[{ts}] {msg} — {(now - start):.1f}s")
    else:
        print(f"[{ts}] {msg}")
    return now

# ----------------------------
# Status system (like flood example)
# ----------------------------
VERBOSE = True
MIN_DISPLAY_SEC = 1
_last_status_time = None

def _wait_for_previous(min_seconds: float):
    global _last_status_time
    if _last_status_time is not None:
        elapsed = time.time() - _last_status_time
        if elapsed < min_seconds:
            time.sleep(min_seconds - elapsed)

def status(msg: str, *, done: bool=False, min_seconds: float = MIN_DISPLAY_SEC):
    if not VERBOSE:
        return
    _wait_for_previous(min_seconds)
    with status_area:
        status_area.clear_output(wait=True)
        style = "color:#333;"
        if done:
            style = "color:#2e7d32;"
        display(HTML(
            f'<div style="font-family:Segoe UI,Arial;font-size:14px;'
            f'padding:6px 8px;border-left:4px solid #888;{style}">{msg}</div>'
        ))
    global _last_status_time
    _last_status_time = time.time()

class StepTimer:
    def __init__(self, start_msg: str, done_label: str = "done", min_seconds: float = MIN_DISPLAY_SEC):
        self.start_msg = start_msg
        self.done_label = done_label
        self.min_seconds = min_seconds
        self.t0 = None
    def __enter__(self):
        self.t0 = time.time()
        status(self.start_msg, done=False, min_seconds=self.min_seconds)
        return self
    def __exit__(self, exc_type, exc, tb):
        dt = time.time() - self.t0
        if exc_type is None:
            status(f"{self.start_msg} — {self.done_label} in {dt:.2f}s", done=True, min_seconds=self.min_seconds)
        else:
            with status_area:
                status_area.clear_output(wait=True)
                display(HTML(
                    f'<div style="font-family:Segoe UI,Arial;font-size:14px;'
                    f'padding:6px 8px;border-left:4px solid #c62828; color:#c62828;">'
                    f'⚠️ Error during: {self.start_msg}<br><code>{exc}</code></div>'
                ))

# Dedicated output area
status_area = widgets.Output()
display(status_area)

def reclassify_clc(code):
    code = str(code)
    if code in ['111', '112']: return '11'
    elif code in ['121', '122', '123', '124']: return '12'
    elif code in ['131', '132', '133']: return '13'
    elif code in ['141', '142']: return '14'
    elif code in ['241','242', '243']: return '24'
    elif code in ['211','212','213','221', '222', '231', '244',
                  '311', '312', '313', '321','323','324','411','421','422']: return code
    elif code in ['331', '333', '334']: return '33'     
    elif code in ['511', '512']: return '51'
    elif code in ['521', '522', '523']: return '52'
    else: return None

with StepTimer("Reading and reclassifying..."):
    gdf_main = gpd.read_file(input_shp)
    gdf_main['clc_l3'] = gdf_main['Code_18'].apply(reclassify_clc)
    gdf_main = gdf_main.dropna(subset=['clc_l3'])

    for clc_val in gdf_main['clc_l3'].unique():
        subset = gdf_main[gdf_main['clc_l3'] == clc_val]
        subset.to_file(os.path.join(paths['split_outputs'], f"CLC_Class_{clc_val}.gpkg"), driver="GPKG")
        status(f"Exported Class: {clc_val}")



Output()

## Step 2: Spatial Filtering (Inner Buffering & Square Creation)

To ensure spectral purity, we:

Apply a -100m inner buffer to avoid "edge effects" near class boundaries.

Generate a 40x40m square (20m radius) training site at the centroid of the safe zone.


In [5]:
# 5. Change the inner buffer distance as per need
# Dedicated output area
status_area = widgets.Output()
display(status_area)

INNER_BUFFER_DIST = -100 #in m
SQUARE_HALF_SIDE = 20  #one side distance from point, total buffer distance 40m
AREA_FILTER = 100 #change the area filter to remove tiny polygons

all_files = [f for f in os.listdir(paths['split_outputs']) if f.endswith(('.shp', '.gpkg'))]

with StepTimer("Applying inner buffer and creating squares..."):
    for file_name in all_files:
        gdf = gpd.read_file(os.path.join(paths['split_outputs'], file_name))
        if gdf.empty: continue
        if gdf.crs.is_geographic: gdf = gdf.to_crs("EPSG:3035")
        
        # 100m Inner Buffer
        gdf['inner_geom'] = gdf.geometry.buffer(INNER_BUFFER_DIST)
        gdf = gdf[~gdf['inner_geom'].is_empty & (gdf['inner_geom'].area > AREA_FILTER)].copy()  #change the area here
    
        if not gdf.empty:
            # Create Square Site
            gdf['centroid_pt'] = gdf['inner_geom'].centroid
            gdf['square_geom'] = gdf['centroid_pt'].buffer(SQUARE_HALF_SIDE, cap_style=3)
            
            # Keep only sites fully within the safe zone
            within_mask = gdf.apply(lambda row: row['square_geom'].within(row['inner_geom']), axis=1)
            gdf_final = gdf[within_mask].set_geometry('square_geom').drop(columns=['inner_geom', 'centroid_pt', 'geometry'])
            gdf_final = gdf_final.rename_geometry('geometry')
    
            gdf_final.to_file(os.path.join(paths['processed_poly'], f"processed_{file_name}"))
        print(f"Processed: {file_name} ({len(gdf_final)} features)")

Output()

Processed: CLC_Class_11.gpkg (190 features)
Processed: CLC_Class_12.gpkg (101 features)
Processed: CLC_Class_13.gpkg (63 features)
Processed: CLC_Class_14.gpkg (34 features)
Processed: CLC_Class_211.gpkg (224 features)
Processed: CLC_Class_212.gpkg (211 features)
Processed: CLC_Class_213.gpkg (17 features)
Processed: CLC_Class_221.gpkg (27 features)
Processed: CLC_Class_222.gpkg (191 features)
Processed: CLC_Class_231.gpkg (134 features)
Processed: CLC_Class_24.gpkg (181 features)
Processed: CLC_Class_244.gpkg (221 features)
Processed: CLC_Class_311.gpkg (309 features)
Processed: CLC_Class_312.gpkg (115 features)
Processed: CLC_Class_313.gpkg (53 features)
Processed: CLC_Class_321.gpkg (88 features)
Processed: CLC_Class_323.gpkg (272 features)
Processed: CLC_Class_324.gpkg (223 features)
Processed: CLC_Class_33.gpkg (27 features)
Processed: CLC_Class_411.gpkg (7 features)
Processed: CLC_Class_421.gpkg (1 features)
Processed: CLC_Class_422.gpkg (1 features)
Processed: CLC_Class_51.gpkg 

## Step 3: Merging Processed Datasets
We merge all individual class files back into a single GeoPackage for final spectral analysis.

In [6]:
# 6. Merging Processed Datasets
status_area = widgets.Output()
display(status_area)
with StepTimer("Merging datasets..."):
    merge_files = [f for f in os.listdir(paths['processed_poly']) if f.endswith('.gpkg')]
    gdf_list = [gpd.read_file(os.path.join(paths['processed_poly'], f)) for f in merge_files]
    
    if gdf_list:
        merged_gdf = pd.concat(gdf_list, ignore_index=True)
        merged_gdf = gpd.GeoDataFrame(merged_gdf, crs=gdf_list[0].crs)
        merged_path = os.path.join(paths['merged_reclassified'], "Merged_CLC_reclassified_Classes.gpkg")
        merged_gdf.to_file(merged_path, driver="GPKG")
        print(f"Successfully merged {len(gdf_list)} files.")

Output()

Successfully merged 24 files.


In [19]:
# 7. Feature extraction + AI-assisted ranking ----

# =========================
# CONFIG
# =========================
USERNAME = "francisco.domingues@uab.es"
PASSWORD = "19@Greenday81"

credentials = Credentials(USERNAME, PASSWORD)

export_dir = os.path.join("C:\\Users\\fdomingues\\cop_t24\\CLC_sampling_process_SPAIN", "exports")
os.makedirs(export_dir, exist_ok=True)

# AOI (same bbox you already use)
minx, miny, maxx, maxy = bbox  # <- from your earlier input
geom_wkt = box(minx, miny, maxx, maxy).wkt

# Seasons
SEASONS = {
    "winter": ("2024-12-01", "2025-02-28"),
    "spring": ("2025-03-01", "2025-05-31"),
    "summer": ("2025-06-01", "2025-08-31"),
    "autumn": ("2025-09-01", "2025-11-30")
}

# =========================
# FUNCTION: compute NDVI
# =========================
def compute_ndvi_array(b4_path, b8_path, scl_path):

    with rasterio.open(b4_path) as red:
        red_arr = red.read(1).astype(float)
        profile = red.profile.copy()
        transform = red.transform

    with rasterio.open(b8_path) as nir:
        nir_arr = nir.read(1).astype(float)

    with rasterio.open(scl_path) as scl_src:
        scl_arr = scl_src.read(1)

        # ✅ simple upscaling (20m → 10m)
        scale = int(red_arr.shape[0] / scl_arr.shape[0])
        scl_resampled = scl_arr.repeat(scale, axis=0).repeat(scale, axis=1)
        scl_resampled = scl_resampled[:red_arr.shape[0], :red_arr.shape[1]]

    # cloud mask
    cloud_mask = np.isin(scl_resampled, [3, 8, 9, 10])

    ndvi = (nir_arr - red_arr) / (nir_arr + red_arr + 1e-6)
    ndvi[cloud_mask] = np.nan

    return ndvi, profile
# =========================
# MAIN LOOP
# =========================
    
MAX_SCENES = 6   # try 5–10

for season, (start_date, end_date) in SEASONS.items():

    print(f"\n🔍 Processing {season}...")

    # -------------------------
    # Query Sentinel-2
    # -------------------------
    features = list(query_features(
        "SENTINEL-2",
        {
            "contentDateStartGe": start_date,
            "contentDateStartLe": end_date,
            "productType": "S2MSI2A",
            "cloudCover": [0, 40],
            "geometry": geom_wkt
        }
    ))

    if len(features) == 0:
        print(f"❌ No data for {season}")
        continue

    # sort by cloud cover
    def get_cloud_cover(f):
        try:
            return f.get("cloudCover", 100)
        except:
            return 100
    
    
    features_sorted = sorted(features, key=get_cloud_cover)
        
    def get_tile_id(f):
    
        # try known fields
        for key in ["title", "name", "id", "productIdentifier"]:
            if key in f:
                parts = f[key].split("_")
                for p in parts:
                    if p.startswith("T") and len(p) == 6:
                        return p
    
        # fallback
        return "UNKNOWN"
    
    # get most frequent tile
    tiles = [get_tile_id(f) for f in features_sorted]
    
    from collections import Counter
    main_tile = Counter(tiles).most_common(1)[0][0]
    
    # keep only scenes from that tile
    candidates = [f for f in features_sorted if get_tile_id(f) == main_tile][:MAX_SCENES] 
    
    #candidates = features_sorted[:MAX_SCENES]

    ndvi_stack = []
    profile_ref = None

    # -------------------------
    # PROCESS EACH SCENE
    # -------------------------
    for i, candidate in enumerate(candidates):

        print(f"   ↳ Scene {i+1}/{len(candidates)}")

        out_dir = os.path.join(export_dir, f"{season}_tmp_{i}")
        os.makedirs(out_dir, exist_ok=True)

        list(download_features(
            [candidate],
            out_dir,
            {"credentials": credentials}
        ))

        # extract ZIP
        for f in os.listdir(out_dir):
            if f.endswith(".zip"):
                zip_path = os.path.join(out_dir, f)
                
                with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                    zip_ref.extractall(out_dir)
                
                # ✅ DELETE zip immediately
                os.remove(zip_path)

        # find SAFE folder
        safe_dirs = [
            os.path.join(out_dir, d)
            for d in os.listdir(out_dir)
            if d.endswith(".SAFE")
        ]

        if not safe_dirs:
            continue

        safe_dir = safe_dirs[0]

        # find bands
        b4 = b8 = scl = None

        for root, _, files in os.walk(safe_dir):
            for f in files:
                full_path = os.path.join(root, f)

                if "R10m" in root:
                    if "_B04_10m.jp2" in f:
                        b4 = full_path
                    elif "_B08_10m.jp2" in f:
                        b8 = full_path

                if "R20m" in root and "_SCL_20m.jp2" in f:
                    scl = full_path

        if not (b4 and b8 and scl):
            print("   ❌ Missing bands → skipping")
            continue

        # NDVI
        ndvi, profile = compute_ndvi_array(b4, b8, scl)

        if np.isnan(ndvi).all():
            print("   ❌ all pixels masked")
            continue

        ndvi_stack.append(ndvi)

        if profile_ref is None:
            profile_ref = profile

        print(f"   ✅ valid pixels: {np.sum(np.isfinite(ndvi))/ndvi.size:.2f}")

    # -------------------------
    # COMPOSITE
    # -------------------------
    if len(ndvi_stack) == 0:
        print(f"❌ No valid NDVI for {season}")
        continue

    print(f"✅ Combining {len(ndvi_stack)} scenes...")

    stack = np.stack(ndvi_stack)
    ndvi_median = np.nanmedian(stack, axis=0)

    ndvi_out = os.path.join(export_dir, f"ndvi_{season}.tif")

    profile_ref.update(
        driver="GTiff",
        dtype=rasterio.float32,
        count=1,
        compress="lzw"
    )

    with rasterio.open(ndvi_out, 'w', **profile_ref) as dst:
        dst.write(ndvi_median.astype(np.float32), 1)

    print(f"✅ FINAL NDVI exported: {ndvi_out}")


print("🧹 Cleaning temporary files...")

for i in range(MAX_SCENES):
    tmp_folder = os.path.join(export_dir, f"{season}_tmp_{i}")

    if os.path.exists(tmp_folder):
        shutil.rmtree(tmp_folder, ignore_errors=True)
        print(f"   ✅ Deleted: {tmp_folder}")

# Paths from your script
#root_dir = r'C:\Users\fdomingues\cop_t24\CLC_sampling_process_SPAIN'  # keep as in your file
#raster_ndvi_path = os.path.join(root_dir, 'NDVI_Tile_29SQB_2018_spain.tif')  # NDVI raster (your script)  # ⟵ uses your NDVI input
merged_path = os.path.join(root_dir, 'merged_reclassified', 'Merged_CLC_reclassified_Classes.gpkg')  # ⟵ your merge output
#export_dir = os.path.join(root_dir, 'exports')

# OPTIONAL: local multi-band Sentinel-2 stack (same CRS/resolution as NDVI)
# If you have a 6-band stack (B2,B3,B4,B8,B11,B12), set this path; else leave as None.
raster_s2_stack_path = None  # e.g., os.path.join(root_dir, 'S2_L2A_2018_stack.tif')

# Quality scoring weights (tune if you like)
W = {
    "ndvi_std": 0.35,          # lower is better
    "ndvi_range": 0.15,        # narrower p90-p10 is better
    "s2_band_std": 0.25,       # lower multi-band std is better
    "interior_margin": 0.15,   # more distance to boundary is better
    "iso_forest": 0.10         # anomaly score -> lower is better
}

# Distance threshold for spatial dispersion (meters)
MIN_SPACING = 500

# Default purity gates (adaptive top-up will relax these)
STDEV_LIMITS = [0.06, 0.08, 0.10]  # your original 0.06 first, then looser
#INNER_BUFFERS = [-100, -60, -40, -20]  # meters
#SQUARE_HALF_SIDES = [20, 16, 12, 10]   # meters -> squares 40, 32, 24, 20 m

INNER_BUFFERS = [-60, -40, -20, -10, 0]   # added 0 as a last-resort fallback
SQUARE_HALF_SIDES = [16, 12, 10, 8]       # 32m, 24m, 20m, 16m squares

# Helper: normalize feature to 0..1 within each class (robust to outliers)
def robust_minmax(x):
    q1, q99 = np.nanpercentile(x, [1, 99])
    return np.clip((x - q1) / max(1e-9, (q99 - q1)), 0, 1)

# Greedy spacing filter
def enforce_min_spacing(gdf, min_dist):
    selected = []
    tree = None
    try:
        from shapely.strtree import STRtree
        tree = STRtree(gdf.geometry.values)
    except Exception:
        # fallback without spatial index (slower)
        pass

    for i, geom in enumerate(gdf.geometry.values):
        if not selected:
            selected.append(i)
            continue
        ok = True
        for j in selected:
            if geom.distance(gdf.geometry.values[j]) < min_dist:
                ok = False
                break
        if ok:
            selected.append(i)
    return gdf.iloc[selected]
print("Finishing setting rankings!")


🔍 Processing winter...
   ↳ Scene 1/6
   ✅ valid pixels: 0.88
   ↳ Scene 2/6
   ✅ valid pixels: 0.71
   ↳ Scene 3/6
   ✅ valid pixels: 0.71
   ↳ Scene 4/6
   ✅ valid pixels: 0.83
   ↳ Scene 5/6
   ✅ valid pixels: 0.97
   ↳ Scene 6/6
   ✅ valid pixels: 0.89
✅ Combining 6 scenes...
✅ FINAL NDVI exported: C:\Users\fdomingues\cop_t24\CLC_sampling_process_SPAIN\exports\ndvi_winter.tif

🔍 Processing spring...
   ↳ Scene 1/6
   ✅ valid pixels: 0.55
   ↳ Scene 2/6
   ✅ valid pixels: 0.82
   ↳ Scene 3/6
   ✅ valid pixels: 0.65
   ↳ Scene 4/6
   ✅ valid pixels: 0.92
   ↳ Scene 5/6
   ✅ valid pixels: 0.87
   ↳ Scene 6/6
   ✅ valid pixels: 0.97
✅ Combining 6 scenes...
✅ FINAL NDVI exported: C:\Users\fdomingues\cop_t24\CLC_sampling_process_SPAIN\exports\ndvi_spring.tif

🔍 Processing summer...
   ↳ Scene 1/6
   ✅ valid pixels: 0.77
   ↳ Scene 2/6
   ✅ valid pixels: 0.95
   ↳ Scene 3/6
   ✅ valid pixels: 0.96
   ↳ Scene 4/6
   ✅ valid pixels: 0.89
   ↳ Scene 5/6
   ✅ valid pixels: 0.89
   ↳ Scene 6/

NameError: name 'shutil' is not defined

## Step 4: Re‑generate candidate squares adaptively (per class)

We’ll rebuild the candidate squares inside your merged file, trying multiple inner buffers and square sizes so that narrow polygons still get a chance. (replaces the single –100 m buffer + 40 m square.)

In [21]:
# 8. Re‑generate candidate squares adaptively (per class) ----
# REQUIREMENTS: StepTimer, status(), INNER_BUFFERS, SQUARE_HALF_SIDES, merged_path must exist

status_area = widgets.Output()
display(status_area)

with StepTimer("Re‑generating candidate squares adaptively (per class)..."):

    # Uses the paths you already defined in your notebook:
    # input_shp, raster_path
    
    assert os.path.exists(input_shp), f"CLC not found: {input_shp}"
    assert os.path.exists(raster_path), f"NDVI not found: {raster_path}"
    
    gdf_src0 = gpd.read_file(input_shp)
    #print("CLC: features =", len(gdf_src0), "CRS =", gdf_src0.crs)
    
    with rasterio.open(raster_path) as src:
        r_crs = src.crs
        r_bounds = src.bounds
        r_bbox = box(*r_bounds)
    #print("NDVI: CRS =", r_crs, "bounds =", tuple(round(v,3) for v in r_bounds))
    
    # Reproject CLC to the raster CRS to test intersection with the NDVI coverage
    gdf_src_r = gdf_src0.to_crs(r_crs)
    intersects = gdf_src_r.intersects(r_bbox)
    print("CLC polygons intersecting the NDVI extent:", int(intersects.sum()))
    if intersects.sum() == 0:
        raise RuntimeError("Your NDVI raster does not cover the CLC area (or CRS mismatch). Check the NDVI tile and AOI.")
    
    # 1) Read CLC and add clc_l3 if missing
    gdf_clc = gpd.read_file(input_shp)
    if 'clc_l3' not in gdf_clc.columns:
        source_code_col = 'Code_18' if 'Code_18' in gdf_clc.columns else None
        if not source_code_col:
            raise KeyError("Could not find the CLC code column (e.g., 'Code_18').")
        gdf_clc['clc_l3'] = gdf_clc[source_code_col].apply(reclassify_clc)
    gdf_clc = gdf_clc.dropna(subset=['clc_l3']).copy()
    
    # 2) Clip to the NDVI raster extent (in the raster CRS)
    with rasterio.open(raster_path) as src:
        r_crs = src.crs
        r_bbox = box(*src.bounds)
    
    gdf_clc_r = gdf_clc.to_crs(r_crs)
    gdf_clc_r = gdf_clc_r[gdf_clc_r.intersects(r_bbox)].copy()
    if gdf_clc_r.empty:
        raise RuntimeError("After clipping to NDVI extent, no CLC polygons remain. Verify the NDVI tile and your AOI.")
    
    # 3) Project to a metric CRS for buffering (EPSG:3035 is good for Europe)
    gdf_clc_m = gdf_clc_r.to_crs("EPSG:3035")
    
    print(f"CLC (clipped to NDVI): {len(gdf_clc_m)} polygons across {gdf_clc_m['clc_l3'].nunique()} classes.")
    # 0) Read and sanity-check merged polygons
    if not os.path.exists(merged_path):
        raise FileNotFoundError(f"Merged file not found at:\n{merged_path}\nRun the merge step first.")

    merged_gdf = gpd.read_file(merged_path)  # must contain 'clc_l3'
    if merged_gdf.empty:
        raise RuntimeError("Merged dataset is empty. Step 2 likely removed everything—relax buffers/square size.")

    if "clc_l3" not in merged_gdf.columns:
        raise KeyError("Column 'clc_l3' not found in merged_gdf. Ensure reclassification step preserved it.")

    def make_square_from_point(pt, half_side):
        x, y = pt.x, pt.y
        return box(x - half_side, y - half_side, x + half_side, y + half_side)

    def make_candidates_for_class(gdf_class, inner_buffers, half_sides):
        rows = []
        gdfc = gdf_class.copy()
        # Heal invalid geometries to avoid errors in negative buffer
        gdfc['geometry'] = gdfc.geometry.buffer(0)

        for ib in inner_buffers:
            safe = gdfc.copy()
            safe["safe_geom"] = safe.geometry.buffer(ib)
            safe = safe[(~safe["safe_geom"].is_empty) & (safe["safe_geom"].is_valid)].copy()
            if safe.empty:
                continue

            cent = safe["safe_geom"].centroid

            for hs in half_sides:
                squares = cent.apply(lambda p: make_square_from_point(p, hs))
                # Keep only squares fully within the safe area
                mask = [sq.within(sg) for sq, sg in zip(squares.values, safe["safe_geom"].values)]
                if not any(mask):
                    continue

                safe2 = safe.loc[mask].copy()
                safe2["geometry"]     = squares.loc[safe2.index].values
                safe2["ib_m"]         = ib
                safe2["half_side_m"]  = hs
                safe2 = safe2.drop(columns=["safe_geom"])
                rows.append(safe2)

        if rows:
            return pd.concat(rows, ignore_index=True)
        # Return empty GeoDataFrame with expected schema
        return gpd.GeoDataFrame(columns=list(gdf_class.columns)+["ib_m","half_side_m"],
                                geometry="geometry", crs=gdf_class.crs)

    candidates = []
    report = []
    for cl, g in gdf_clc_m.groupby("clc_l3"):
        c = make_candidates_for_class(g, INNER_BUFFERS, SQUARE_HALF_SIDES)
        n = 0 if c is None or c.empty else len(c)
        report.append((cl, n))
        if n > 0:
            if "clc_l3" not in c.columns:
                c = c.assign(clc_l3=cl)
            candidates.append(c)

    if not candidates:
        msg = "\n".join([f"  - Class {cl}: {n} candidates" for cl, n in report])
        raise RuntimeError("Still no candidates. Try INNER_BUFFERS=[-40,-20,-10,0] and SQUARE_HALF_SIDES=[12,10,8,6].\n" + msg)

    candidates_gdf = gpd.GeoDataFrame(pd.concat(candidates, ignore_index=True), crs=gdf_clc_m.crs)



    
    total = len(candidates_gdf)
    status(f"✅ Candidate squares generated: {total}")
    print("Per-class candidate counts:")
    display(candidates_gdf["clc_l3"].value_counts().sort_index())
    

Output()

CLC polygons intersecting the NDVI extent: 4538
CLC (clipped to NDVI): 4081 polygons across 24 classes.
Per-class candidate counts:


clc_l3
11     4493
12     2296
13     1415
14      822
211    5172
212    4892
213     340
221     656
222    4307
231    3209
24     4623
244    5370
311    7692
312    2745
313    1309
321    2043
323    6856
324    5104
33      694
411     178
421      20
422      20
51      842
52       20
Name: count, dtype: int64

## Feature extraction (NDVI + optional S2)

We compute NDVI stats using your NDVI raster; if you provide a local S2 band stack, we also compute per‑band standard deviations for additional purity signals.

In [ ]:
# 9. Feature extraction (NDVI + optional S2)  — REVISED, robust
status_area = widgets.Output()
display(status_area)

SHRINK_FACTORS = [1.0, 0.9, 0.8, 0.7, 0.6]

# =========================================================
# SEASONAL NDVI INPUTS
# =========================================================
export_dir = os.path.join("C:\\Users\\fdomingues\\cop_t24\\CLC_sampling_process_SPAIN", "exports")

SEASONAL_NDVI = {
    "winter": os.path.join(export_dir, "ndvi_winter.tif"),
    "spring": os.path.join(export_dir, "ndvi_spring.tif"),
    "summer": os.path.join(export_dir, "ndvi_summer.tif"),
    "autumn": os.path.join(export_dir, "ndvi_autumn.tif")
}

all_season_results = []

with StepTimer("Extracting features (NDVI)..."):

    for season_name, raster_ndvi_path in SEASONAL_NDVI.items():

        print("\n" + "="*60)
        print(f"PROCESSING SEASON: {season_name.upper()}")
        print("="*60)
            
        # ================================================================
        # STEP 4 — OPTIMIZED NDVI FEATURE EXTRACTION & SAMPLE SELECTION
        # ================================================================
        
        # --------------------------------------------------------------
        # 4.0 LOAD CANDIDATES FROM PREVIOUS STEP
        # --------------------------------------------------------------
        print("Initial candidates:", len(candidates_gdf))
        
        
        # --------------------------------------------------------------
        # 4.1 REDUCE POLYGONS BEFORE NDVI (CRITICAL FOR SPEED)
        # --------------------------------------------------------------
        filtered = []
        for cl, g in candidates_gdf.groupby("clc_l3"):
            if len(g) > 800:       # limit oversampled classes
                g = g.sample(800, random_state=42)
            filtered.append(g)
        
        candidates_gdf = gpd.GeoDataFrame(pd.concat(filtered), crs=candidates_gdf.crs)
        print("Reduced candidates:", len(candidates_gdf))
        
        
        # --------------------------------------------------------------
        # 4.2 FAST NDVI MEAN & STD USING zonal_stats
        # --------------------------------------------------------------
    
        # Reproject to NDVI CRS    
        with rasterio.open(raster_ndvi_path) as src:
            ndvi_crs = src.crs

        # ✅ FIX: fallback if CRS missing
        if ndvi_crs is None:
            print("⚠️ NDVI raster has no CRS — assigning EPSG:3035")
            ndvi_crs = "EPSG:3035"
        
        # ✅ FIX: ensure candidates have CRS
        if candidates_gdf.crs is None:
            print("⚠️ candidates_gdf has no CRS — assigning EPSG:3035")
            candidates_gdf = candidates_gdf.set_crs("EPSG:3035")

        candidates_ndvi = candidates_gdf.to_crs(ndvi_crs)    
        
        if candidates_gdf.crs is None:
            raise ValueError("❌ candidates_gdf has no CRS. You must assign it earlier.")        
        print("Candidates CRS:", candidates_gdf.crs)
        print("NDVI CRS:", ndvi_crs)

        # Extract fast features
        with rasterio.open(raster_ndvi_path) as src2:
            ndvi_nodata = src2.nodata
    
        zs = zonal_stats(
            candidates_ndvi,
            raster_ndvi_path,
            stats=["mean", "std"],
            nodata=ndvi_nodata
        )
        
        mean_std_df = pd.DataFrame(zs)
        mean_std_df.rename(columns={"mean": "ndvi_mean", "std": "ndvi_std"}, inplace=True)
    
        print(mean_std_df.describe())
        
        feat_df = candidates_gdf.reset_index(drop=True).copy()
        feat_df["ndvi_mean"] = mean_std_df["ndvi_mean"]
        feat_df["ndvi_std"]  = mean_std_df["ndvi_std"]
        
        print("NDVI mean/std extracted.")
        
        
        # --------------------------------------------------------------
        # 4.3 SELECT FINALISTS FOR DEEP ANALYSIS
        #     (smaller set -> fast percentiles)
        # --------------------------------------------------------------
        FINALIST_LIMIT = 250     # good balance: purity + speed
        
        finalists = []
        for cl, g in feat_df.groupby("clc_l3"):
            g = g.sort_values("ndvi_std", ascending=True)
            finalists.append(g.head(min(FINALIST_LIMIT, len(g))))
        
        finalists = gpd.GeoDataFrame(pd.concat(finalists), crs=feat_df.crs)
        print("Finalists:", len(finalists))
        
        
        # --------------------------------------------------------------
        # 4.4 NDVI PERCENTILES (10, 50, 90) ONLY FOR FINALISTS
        # --------------------------------------------------------------
        t0 = time.time()
        
        def compute_percentiles_single(geom, raster_path):
            with rasterio.open(raster_path) as src:
                try:
                    arr, _ = mask(src, [mapping(geom)], crop=True, filled=False)
                except:
                    return np.nan, np.nan, np.nan
        
                band = arr[0]
        
                vals = band.compressed() if np.ma.isMaskedArray(band) else band.ravel()
                vals = vals[np.isfinite(vals)]
        
                vals = vals[vals > -0.1]   # remove invalid NDVI
        
                if len(vals) < 10:   # important threshold
                    return np.nan, np.nan, np.nan
        
                return (
                    float(np.nanpercentile(vals, 10)),
                    float(np.nanpercentile(vals, 50)),
                    float(np.nanpercentile(vals, 90))
                )
        
        #percentiles = [compute_percentiles_single(g, raster_ndvi_path) for g in finalists.geometry]
        
        finalists_ndvi = finalists.to_crs(ndvi_crs)
        
        percentiles = [
            compute_percentiles_single(g, raster_ndvi_path)
            for g in finalists_ndvi.geometry
        ]    
        
        finalists["ndvi_p10"]   = [p[0] for p in percentiles]
        finalists["ndvi_p50"]   = [p[1] for p in percentiles]
        finalists["ndvi_p90"]   = [p[2] for p in percentiles]
        finalists["ndvi_range"] = finalists["ndvi_p90"] - finalists["ndvi_p10"]
    
        print(f"Percentiles computed.  ⏱️ Took: {time.time() - t0:.2f} sec")
    
        # --- FORCE ALL NDVI COLUMNS TO PURE NUMERIC ---
        num_cols = ["ndvi_mean","ndvi_std","ndvi_p10","ndvi_p50","ndvi_p90","ndvi_range"]
        for c in num_cols:
            finalists[c] = pd.to_numeric(finalists[c], errors="coerce")
            
        # SPEED-UP: simplify original CLC polygons before union (ROBUST)
        gdf_clc_simpl = gdf_clc_m.copy()
        
        # Ensure SAME CRS as finalists (EPSG:3035)
        if gdf_clc_simpl.crs != finalists.crs:
            gdf_clc_simpl = gdf_clc_simpl.to_crs(finalists.crs)
            
        # 1. Fix invalid geometries properly
        gdf_clc_simpl["geometry"] = gdf_clc_simpl["geometry"].apply(make_valid)
        
        # 2. Remove empty geometries
        gdf_clc_simpl = gdf_clc_simpl[~gdf_clc_simpl.is_empty]
        
        # 3. Keep only polygons (avoid GeometryCollections etc.)
        gdf_clc_simpl = gdf_clc_simpl[
            gdf_clc_simpl.geometry.type.isin(["Polygon", "MultiPolygon"])
        ]
        
        # 4. Explode multipolygons (IMPORTANT)
        gdf_clc_simpl = gdf_clc_simpl.explode(index_parts=False)
        
        # 5. Simplify AFTER fixing
        gdf_clc_simpl["geometry"] = gdf_clc_simpl.geometry.simplify(
            50, preserve_topology=True
        )
        
        # 6. Final safety buffer
        gdf_clc_simpl["geometry"] = gdf_clc_simpl["geometry"].buffer(0)
    
        # --------------------------------------------------------------
        # 4.5 INTERIOR MARGIN (distance from square to original CLC)
        # --------------------------------------------------------------
        if finalists.crs.is_geographic:
            finalists = finalists.to_crs("EPSG:3035")
        
        def safe_union(geoms):
            try:
                return unary_union(geoms)
            except Exception:
                # fallback (slower but avoids crash)
                return gpd.GeoSeries(geoms).unary_union
        
        # Use ORIGINAL geometries (NOT simplified!)
        gdf_clc_for_union = gdf_clc_m.copy()
        
        # Ensure CRS matches finalists
        if gdf_clc_for_union.crs != finalists.crs:
            gdf_clc_for_union = gdf_clc_for_union.to_crs(finalists.crs)
        
        orig_union_by_class = {
            cl: safe_union(
                gdf_clc_for_union.loc[gdf_clc_for_union["clc_l3"] == cl, "geometry"]
            )
            for cl in gdf_clc_for_union["clc_l3"].unique()
        }
    
        def interior_margin(row):
            outer = orig_union_by_class[row["clc_l3"]]
            sq = row["geometry"]
        
            if outer is None or outer.is_empty:
                return 0.0
        
            # ✅ Use centroid instead of full polygon
            pt = sq.centroid
        
            # ✅ Much more robust than contains(sq)
            if not outer.contains(pt):
                return 0.0
        
            # ✅ Distance in meters (correct CRS assumed)
            return pt.distance(outer.boundary)
            
        finalists["interior_margin_m"] = finalists.apply(interior_margin, axis=1)
        print("Interior margin computed.")
        
        
        # --------------------------------------------------------------
        # 4.6 ISOLATION FOREST (per class)
        # --------------------------------------------------------------
        finalists["iso_penalty"] = np.nan
        
        for cl, g in finalists.groupby("clc_l3"):
            if len(g) < 20:
                finalists.loc[g.index, "iso_penalty"] = 0
                continue
            
            numeric_cols = ["ndvi_mean","ndvi_std","ndvi_range","interior_margin_m"]
            
            X = g[numeric_cols].copy()
            
            # Replace inf with nan
            X = X.replace([np.inf, -np.inf], np.nan)
            
            # Fill only using medians of THESE columns, not all g columns
            X = X.fillna(X.median(numeric_only=True))
            
            iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
            iso.fit(X)
            scores = iso.score_samples(X)
            
            norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
            finalists.loc[g.index, "iso_penalty"] = 1 - norm
        
        print("Isolation Forest done.")
        
        
        # --------------------------------------------------------------
        # 4.7 QUALITY SCORE
        # --------------------------------------------------------------
        def robust_minmax(x):
            q1, q99 = np.nanpercentile(x, [1, 99])
            return np.clip((x - q1) / (q99 - q1 + 1e-9), 0, 1)
        
        W = {"ndvi_std":0.35, "ndvi_range":0.15, "margin":0.25, "iso":0.25}
        
        ranked = []
        for cl, g in finalists.groupby("clc_l3"):
            g = g.copy()
            g["ndvi_std_n"]   = 1 - robust_minmax(g["ndvi_std"])
            g["ndvi_range_n"] = 1 - robust_minmax(g["ndvi_range"])
            g["margin_n"]     =     robust_minmax(g["interior_margin_m"])
            g["iso_n"]        = 1 - robust_minmax(g["iso_penalty"])
            
            g["quality_score"] = (
                W["ndvi_std"]*g["ndvi_std_n"] +
                W["ndvi_range"]*g["ndvi_range_n"] +
                W["margin"]*g["margin_n"] +
                W["iso"]*g["iso_n"]
            ) * 100
            
            g["quality_score"] = g["quality_score"].fillna(0)
            
            ranked.append(g)
        
        ranked = gpd.GeoDataFrame(pd.concat(ranked), crs=finalists.crs)
        ranked = ranked.sort_values(["clc_l3","quality_score"], ascending=[True, False])
        print("Ranking done. Total:", len(ranked))
    
        ranked["sample_source"] = "original"
        
        # --------------------------------------------------------------
        # 4.7b CLASS-ADAPTIVE NDVI TARGETS
        # --------------------------------------------------------------
        class_ndvi_target = {}
        class_ndvi_std = {}
        
        grouped = ranked.groupby("clc_l3")["ndvi_mean"]
        
        for cl, values in grouped:
        
            values = values.dropna()
        
            if len(values) < 10:
                class_ndvi_target[cl] = values.median() if len(values) > 0 else 0
                class_ndvi_std[cl] = values.std() if len(values) > 1 else 0.05
                continue
        
            # remove extremes (robust)
            q_low = values.quantile(0.1)
            q_high = values.quantile(0.9)
            values_f = values[(values >= q_low) & (values <= q_high)]
        
            # class-specific logic
            if cl in [51, 52]:  # water
                target = values_f.quantile(0.25)
            elif cl in [311, 312, 313]:  # forest
                target = values_f.quantile(0.75)
            else:
                target = values_f.median()
        
            class_ndvi_target[cl] = target
            class_ndvi_std[cl] = values_f.std() if len(values_f) > 1 else 0.05
        
        print("NDVI class targets computed.")
        
        # --------------------------------------------------------------
        # 4.8 SELECT TOP 60 PER CLASS WITH ADAPTIVE TOP‑UP
        # --------------------------------------------------------------
        
        TARGET = 60
        MIN_SPACING = 500   # initial spacing requirement
                    
        def enforce_centroid_distance(gdf, min_dist=80):
            selected_idx = []
            selected_centroids = []
        
            for idx, geom in zip(gdf.index, gdf.geometry):
        
                c = geom.centroid
                keep = True
        
                for sc in selected_centroids:
                    if c.distance(sc) < min_dist:
                        keep = False
                        break
        
                if keep:
                    selected_idx.append(idx)
                    selected_centroids.append(c)
        
            return gdf.loc[selected_idx]
        
        def shrink_geometry(gdf, factor):
            gdf = gdf.copy()
            gdf["geometry"] = gdf.geometry.buffer(0)  # clean
            
            gdf["geometry"] = gdf.geometry.scale(
                xfact=factor,
                yfact=factor,
                origin="center"
            )
            return gdf
            
        def generate_random_samples(polygon, n_samples, size, max_attempts=20000):
            samples = []
            minx, miny, maxx, maxy = polygon.bounds
        
            attempts = 0
        
            while len(samples) < n_samples and attempts < max_attempts:
                attempts += 1
        
                x = random.uniform(minx, maxx)
                y = random.uniform(miny, maxy)
        
                half = size / 2
                sq = box(x - half, y - half, x + half, y + half)
        
                # 🔥 LESS STRICT (critical for 411)
                if not sq.intersects(polygon):
                    continue
        
                samples.append(sq)
        
            return samples
            
        final_rows = []
        
        SQUARE_SIZE = 100  # <-- IMPORTANT: match your real sample size
        
        for cl, g in ranked.groupby("clc_l3"):
    
            t_class_start = time.time()
            
            print(f"\nProcessing class {cl}...")
        
            g_sorted = g.sort_values("quality_score", ascending=False)
        
            best_sel = None
        
            for factor in SHRINK_FACTORS:
        
                if factor < 1.0:
                    g_try = g_sorted.copy()
                    g_try["geometry"] = g_try.geometry.scale(
                        xfact=factor,
                        yfact=factor,
                        origin="center"
                    )
                else:
                    g_try = g_sorted
        
                non_overlap = enforce_centroid_distance(g_try, min_dist=80)
        
                if len(non_overlap) >= TARGET:
                    best_sel = non_overlap.head(TARGET)
                    break
        
                if best_sel is None or len(non_overlap) > len(best_sel):
                    best_sel = non_overlap
        
            # restore ORIGINAL geometries
            sel = g_sorted.loc[best_sel.index].copy()
                        
            # ------------------------------------------------------------
            # 🔥 DETERMINISTIC NDVI-OPTIMAL GRID SELECTION (REPLACEMENT)
            # ------------------------------------------------------------
            if len(sel) < TARGET:
            
                needed = TARGET - len(sel)
                print(f"  → Filling {needed} samples using NDVI-optimized grid...")
            
                class_polygon = orig_union_by_class[cl]
                
                # 🔥 shrink polygon so squares fit fully inside
                # ------------------------------------------------------------
                # 🔥 ADAPTIVE SAFE POLYGON
                # ------------------------------------------------------------
                if cl == 411:
                    SAFE_MARGIN = SQUARE_SIZE * 0.25   # MUCH less aggressive
                else:
                    SAFE_MARGIN = SQUARE_SIZE / 2
                
                safe_polygon = class_polygon.buffer(-SAFE_MARGIN)
                
                if safe_polygon.is_empty:
                    safe_polygon = class_polygon.buffer(-SAFE_MARGIN * 0.5)
                
                if safe_polygon.is_empty:
                    safe_polygon = class_polygon
                
                # fallback if polygon collapses (very important)
                if safe_polygon.is_empty:
                    safe_polygon = class_polygon.buffer(-SAFE_MARGIN * 0.5)
                
                if safe_polygon.is_empty:
                    safe_polygon = class_polygon  # last resort
            
    
                # ------------------------------------------------------------
                #  FAST NDVI-AWARE SAMPLE GENERATION (NO GRID)
                # ------------------------------------------------------------
                def generate_random_points_fast(polygon, n, oversample=20):
                
                    minx, miny, maxx, maxy = polygon.bounds
                
                    pts = []
                    tries = n * oversample
                
                    xs = np.random.uniform(minx, maxx, tries)
                    ys = np.random.uniform(miny, maxy, tries)
                
                    poly_buffered = polygon.buffer(20)   # ✅ compute ONCE
                
                    for x, y in zip(xs, ys):
                        p = Point(x, y)
                        if poly_buffered.contains(p):
                            pts.append(p)
                        if len(pts) >= n:
                            break
                
                    return pts
                
                
                def points_to_squares(points, size):
                    half = size / 2
                    return [box(p.x - half, p.y - half, p.x + half, p.y + half) for p in points]
                        
                #  FAST RANDOM POINT GENERATION
                if needed <= 5:
                    factor = 10
                elif needed <= 20:
                    factor = 20
                else:
                    factor = 30
                
                # special case
                if cl == 411:
                    factor *= 4
                
                pts = generate_random_points_fast(safe_polygon, n=needed * factor)
    
                # LIMIT before NDVI (HUGE SPEED BOOST)
                if cl == 411:
                    MAX_GRID = 1200
                else:
                    MAX_GRID = 400
                
                if len(pts) > MAX_GRID:
                    pts = random.sample(pts, MAX_GRID)
                
                if len(pts) == 0:
                    print("  ⚠️ No candidates generated → relaxing constraints")
                
                    # Retry WITHOUT buffer (much less strict)
                    pts = generate_random_points_fast(safe_polygon, n=needed * factor * 2, oversample=50)
                
                    if len(pts) == 0:
                        print("  ❌ Still no candidates — skipping NDVI filter")
                        
                        # fallback: random squares anywhere in bbox
                        minx, miny, maxx, maxy = safe_polygon.bounds
                        pts = [Point(
                            random.uniform(minx, maxx),
                            random.uniform(miny, maxy)
                        ) for _ in range(needed * 5)]
                else:
                    squares = points_to_squares(pts, SQUARE_SIZE * 0.4)
                
                    grid_gdf = gpd.GeoDataFrame(
                        {
                            "clc_l3": [cl] * len(squares),
                            "sample_source": ["generated"] * len(squares),
                        },
                        geometry=squares,
                        crs=sel.crs
                    )
                    
                    # FIX 3 — FILTER VALID GEOMETRIES (VERY IMPORTANT)
                    # STRICT containment (square fully inside class)
                    grid_gdf = grid_gdf[
                        grid_gdf.geometry.within(safe_polygon)
                    ].copy()
                    
                    if len(grid_gdf) == 0:
                        print("  ⚠️ No valid geometries after intersection filter")
                    else:
                        # NDVI starts here
                        grid_ndvi = grid_gdf.to_crs(ndvi_crs)
                            
                    # ------------------------------------------
                    # 2. COMPUTE NDVI FOR GRID
                    # ------------------------------------------
                    #grid_ndvi = grid_gdf.to_crs(ndvi_crs)
            
                    zs_grid = zonal_stats(
                        grid_ndvi,
                        raster_ndvi_path,
                        stats=["mean"],
                        nodata=ndvi_nodata
                    )
            
                    grid_gdf["ndvi_mean"] = [z["mean"] for z in zs_grid]
            
                    # Remove invalid NDVI
                    grid_gdf = grid_gdf[np.isfinite(grid_gdf["ndvi_mean"])]
            
                    # ------------------------------------------
                    # 3. NDVI TARGETING (CLASS-AWARE)
                    # ------------------------------------------
                    target_ndvi = class_ndvi_target.get(cl, 0)
                    std_ndvi = class_ndvi_std.get(cl, 0.05)
            
                    grid_gdf["ndvi_distance"] = (
                        (grid_gdf["ndvi_mean"] - target_ndvi).abs() / (std_ndvi + 1e-6)
                    )
            
                    # 🔥 Sort by BEST NDVI FIRST
                    grid_gdf = grid_gdf.sort_values("ndvi_distance")
            
                    # ------------------------------------------
                    # 4. GREEDY SELECTION WITH SPACING
                    # ------------------------------------------
                    selected_geoms = list(sel.geometry)  # keep existing best ones
                            
                    MIN_DIST = 40
                    
                    if cl == 411:
                        MIN_DIST = 10
            
                    new_selected = []
            
                    for geom in grid_gdf.sort_values("ndvi_distance").geometry:
            
                        c = geom.centroid
            
                        keep = True
                        for existing in selected_geoms:
                            if c.distance(existing.centroid) < MIN_DIST:
                                keep = False
                                break
            
                        if keep:
                            new_selected.append(geom)
                            selected_geoms.append(geom)
            
                        if len(new_selected) >= needed:
                            break
                    
                    if len(new_selected) < needed:
                        print(f"  ⚠️ Only filled {len(new_selected)} of {needed}")
                    
                        # OPTIONAL FIX — FORCE FILL FOR HARD CLASSES (e.g. 411)
                        if cl == 411:
                            print("  → Applying fallback fill (relaxed constraints)")
                    
                            for geom in grid_gdf.geometry:
                                if geom not in new_selected:
                                    new_selected.append(geom)
                                if len(new_selected) >= needed:
                                    break      
                                    
                    # ------------------------------------------
                    # 5. MERGE BACK
                    # ------------------------------------------
                    if len(new_selected) > 0:
                        new_gdf = gpd.GeoDataFrame(
                            {
                                "clc_l3": [cl] * len(new_selected),
                                "sample_source": ["generated"] * len(new_selected),
                            },
                            geometry=new_selected,
                            crs=sel.crs
                        )
            
                        sel = pd.concat([sel, new_gdf], ignore_index=True)
        
            final_rows.append(sel.head(TARGET))
    
            print(f"  ⏱️ Took: {time.time() - t_class_start:.2f} sec")
        
        final = gpd.GeoDataFrame(pd.concat(final_rows), crs=ranked.crs)
        final["season"] = season_name
        #print("\nFinal per class (guaranteed 60 each):")
        print("\nFinal per class (non-overlapping):")
        print(final["clc_l3"].value_counts().sort_index())
        
        
        # --------------------------------------------------------------
        # 4.9 SAVE OUTPUTS
        # --------------------------------------------------------------
        final_out = os.path.join(export_dir,f"best_60_per_class_{season_name}.gpkg")
        diag_csv  = os.path.join(export_dir, f"sampling_diagnostics_{season_name}.csv")
        
        final.to_file(final_out, driver="GPKG")
        
        diag = final.groupby("clc_l3").agg(
            n=("clc_l3","count"),
            q25=("quality_score", lambda x: np.percentile(x,25)),
            q50=("quality_score","median"),
            q75=("quality_score", lambda x: np.percentile(x,75)),
            ndvi_std_med=("ndvi_std","median")
        ).reset_index()
        
        diag.to_csv(diag_csv, index=False)
        all_season_results.append(final)
        print("Saved:", final_out)
        print("Saved:", diag_csv)

# =========================================================
# MERGE ALL SEASONS
# =========================================================

final_4seasons = gpd.GeoDataFrame(
    pd.concat(all_season_results, ignore_index=True),
    crs=all_season_results[0].crs
)

print("\nTOTAL SAMPLES:")
print(len(final_4seasons))

print("\nSamples per season:")
print(final_4seasons["season"].value_counts())

merged_out = os.path.join(
    export_dir,
    "best_60_per_class_ALL_SEASONS.gpkg"
)

final_4seasons.to_file(
    merged_out,
    driver="GPKG"
)

print("Saved merged seasonal dataset:")
print(merged_out)

Output()


PROCESSING SEASON: WINTER
Initial candidates: 65118
Reduced candidates: 15528
⚠️ NDVI raster has no CRS — assigning EPSG:3035
Candidates CRS: EPSG:3035
NDVI CRS: EPSG:3035
       ndvi_mean  ndvi_std
count    15528.0   15528.0
mean         0.0       0.0
std          0.0       0.0
min          0.0       0.0
25%          0.0       0.0
50%          0.0       0.0
75%          0.0       0.0
max          0.0       0.0
NDVI mean/std extracted.
Finalists: 5238
Percentiles computed.  ⏱️ Took: 8.16 sec
Interior margin computed.
Isolation Forest done.
Ranking done. Total: 5238
NDVI class targets computed.

Processing class 11...
  ⏱️ Took: 0.28 sec

Processing class 12...
  ⏱️ Took: 0.20 sec

Processing class 13...
  ⏱️ Took: 0.15 sec

Processing class 14...
  → Filling 13 samples using NDVI-optimized grid...
  ⏱️ Took: 12.47 sec

Processing class 211...
  ⏱️ Took: 0.34 sec

Processing class 212...
  ⏱️ Took: 0.35 sec

Processing class 213...
  → Filling 42 samples using NDVI-optimized grid...
  

In [ ]:
# 10. FINAL PDF GENERATOR — With Image Loading + NDVI Filtering + CLC Metadata
# =====================================================

from rasterio.mask import mask as rio_mask
status_area = widgets.Output()
display(status_area)

with StepTimer("Exporting pdf summary..."):
    
    output_pdf = os.path.join(export_dir, "pdf", "CLC_class_summary.pdf")
    
    # Ensure directory exists
    os.makedirs(os.path.dirname(output_pdf), exist_ok=True)
    
    s2_path = os.path.join(root_dir, "S2_tiles_29SQB_2018.tif")
    assert os.path.exists(s2_path)
    # ============================================================
    # 0. USER-DEFINED CLC METADATA (YOU WILL FILL THIS YOURSELF)
    # ============================================================
    clc_metadata = {
        # ---------------------------------------------------------
        # LEVEL 2 CLASSES (using fallback L3 as agreed)
        # ---------------------------------------------------------
    
        11: {
            "level1": "Artificial surfaces",  #[1](https://whisp-help.earthmap.org/datasets/land-useland-cover/land-cover-corine)
            "level2": "Urban fabric",        #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py).py)
            "level3": "Continuous urban fabric (fallback: 111)",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py)
            "description": "Dense built‑up residential areas with high soil sealing and tightly packed buildings.",  #summary
            "example_folder": "index-clc-111"
        },
    
        12: {
            "level1": "Artificial surfaces",
            "level2": "Industrial, commercial and transport units",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py).py)
            "level3": "Industrial or commercial units (fallback: 121)",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py)
            "description": "Industrial estates, commercial zones or large service facilities dominated by built structures.",
            "example_folder": "index-clc-121"
        },
    
        13: {
            "level1": "Artificial surfaces",
            "level2": "Mine, dump and construction sites",   #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py).py)
            "level3": "Mineral extraction sites (fallback: 131)",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py)
            "description": "Areas undergoing excavation, mining or major construction activities with exposed earth.",
            "example_folder": "index-clc-131"
        },
    
        14: {
            "level1": "Artificial surfaces",
            "level2": "Artificial, non‑agricultural vegetated areas",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py).py)
            "level3": "Green urban areas (fallback: 141)",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py)
            "description": "Parks and managed green spaces surrounded by urban development.",
            "example_folder": "index-clc-141"
        },
    
        24: {
            "level1": "Agricultural areas",   #[1](https://whisp-help.earthmap.org/datasets/land-useland-cover/land-cover-corine)
            "level2": "Heterogeneous agricultural areas",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py).py)
            "level3": "Annual crops associated with permanent crops (fallback: 241)",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py)
            "description": "Mixed agricultural landscapes where annual crops coexist with permanent crops or natural patches.",
            "example_folder": "index-clc-241"
        },
    
        33: {
            "level1": "Forests and semi‑natural areas",  #[1](https://whisp-help.earthmap.org/datasets/land-useland-cover/land-cover-corine)
            "level2": "Open spaces with little or no vegetation",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py).py)
            "level3": "Beaches, dunes, sands (fallback: 331)",     #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py)
            "description": "Areas with extremely sparse vegetation, including dunes, bare soils and sandy surfaces.",
            "example_folder": "index-clc-331"
        },
    
        51: {
            "level1": "Water bodies",  #[1](https://whisp-help.earthmap.org/datasets/land-useland-cover/land-cover-corine)
            "level2": "Inland waters",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py).py)
            "level3": "Water courses (fallback: 511)",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py)
            "description": "Rivers and flowing water channels, natural or artificial.",
            "example_folder": "index-clc-511"
        },
    
        52: {
            "level1": "Water bodies",
            "level2": "Marine waters",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py).py).py)
            "level3": "Coastal lagoons (fallback: 521)",
            "description": "Coastal water bodies partially separated from the sea by natural or artificial barriers.",
            "example_folder": "index-clc-521"
        },
    
        # ---------------------------------------------------------
        # LEVEL 3 CLASSES (direct mapping)
        # ---------------------------------------------------------
    
        211: {
            "level1": "Agricultural areas",
            "level2": "Arable land",
            "level3": "Non‑irrigated arable land",  #[2](https://uab-my.sharepoint.com/personal/1170696_uab_cat/Documents/Archivos%20de%20chat%20de%20Microsoft%C2%A0Copilot/CLC_automated_stepByStep_sampling_FD_UAB_v2%20(2).py).py).py).py)
            "description": "Extensive fields of cereals or dryland crops without irrigation.",
            "example_folder": "index-clc-211"
        },
    
        212: {
            "level1": "Agricultural areas",
            "level2": "Arable land",
            "level3": "Permanently irrigated land",
            "description": "Cultivated fields dependent on permanent irrigation systems.",
            "example_folder": "index-clc-212"
        },
    
        213: {
            "level1": "Agricultural areas",
            "level2": "Arable land",
            "level3": "Rice fields",
            "description": "Terraced or levelled fields flooded seasonally for rice cultivation.",
            "example_folder": "index-clc-213"
        },
    
        221: {
            "level1": "Agricultural areas",
            "level2": "Permanent crops",
            "level3": "Vineyards",
            "description": "Vine plantations arranged in rows for wine or table grape production.",
            "example_folder": "index-clc-221"
        },
    
        222: {
            "level1": "Agricultural areas",
            "level2": "Permanent crops",
            "level3": "Fruit trees and berry plantations",
            "description": "Orchards with fruit species or berry plantations.",
            "example_folder": "index-clc-222"
        },
    
        231: {
            "level1": "Agricultural areas",
            "level2": "Pastures",
            "level3": "Pastures",
            "description": "Grassland used for grazing livestock with regular mowing or grazing.",
            "example_folder": "index-clc-231"
        },
    
        244: {
            "level1": "Agricultural areas",
            "level2": "Heterogeneous agricultural areas",
            "level3": "Agro‑forestry areas",
            "description": "Land where trees and agricultural crops coexist in the same spatial unit.",
            "example_folder": "index-clc-244"
        },
    
        311: {
            "level1": "Forests and semi‑natural areas",
            "level2": "Forests",
            "level3": "Broad‑leaved forest",
            "description": "Deciduous forest stands with a canopy dominated by broad‑leaved species.",
            "example_folder": "index-clc-311"
        },
    
        312: {
            "level1": "Forests and semi‑natural areas",
            "level2": "Forests",
            "level3": "Coniferous forest",
            "description": "Evergreen forests primarily composed of conifer species.",
            "example_folder": "index-clc-312"
        },
    
        313: {
            "level1": "Forests and semi‑natural areas",
            "level2": "Forests",
            "level3": "Mixed forest",
            "description": "Forest stands where neither broad‑leaved nor coniferous species dominate.",
            "example_folder": "index-clc-313"
        },
    
        321: {
            "level1": "Forests and semi‑natural areas",
            "level2": "Scrub and/or herbaceous vegetation associations",
            "level3": "Natural grasslands",
            "description": "Semi‑natural grasslands dominated by herbaceous vegetation with low tree cover.",
            "example_folder": "index-clc-321"
        },
    
        323: {
            "level1": "Forests and semi‑natural areas",
            "level2": "Scrub and/or herbaceous vegetation associations",
            "level3": "Sclerophyllous vegetation",
            "description": "Mediterranean shrubs adapted to drought with hard, leathery leaves.",
            "example_folder": "index-clc-323"
        },
    
        324: {
            "level1": "Forests and semi‑natural areas",
            "level2": "Scrub and/or herbaceous vegetation associations",
            "level3": "Transitional woodland‑shrub",
            "description": "Mosaics of shrubs and scattered trees in transition to forest.",
            "example_folder": "index-clc-324"
        },
    
        411: {
            "level1": "Wetlands",
            "level2": "Inland wetlands",
            "level3": "Inland marshes",
            "description": "Wetlands dominated by reeds, sedges or marsh vegetation with standing water.",
            "example_folder": "index-clc-411"
        },
    
        421: {
            "level1": "Wetlands",
            "level2": "Coastal wetlands",
            "level3": "Salt marshes",
            "description": "Coastal marsh areas periodically flooded by saline or brackish waters.",
            "example_folder": "index-clc-421"
        },
    
        422: {
            "level1": "Wetlands",
            "level2": "Coastal wetlands",
            "level3": "Salines",
            "description": "Salt extraction basins with shallow water tanks for salt production.",
            "example_folder": "index-clc-422"
        },
    }
    
    # =====================================================
    # FIXED PDF GENERATOR — NDVI + S2 + GRID + BORDERS + CRS-FIXED
    # =====================================================
    
    '''
    def put_border(ax, lw=1.2):
        for s in ax.spines.values():
            s.set_edgecolor("black")
            s.set_visible(True)
            s.set_linewidth(lw)
    
    def put_border(fig, axes, pad=0.05):
        boxes = [ax.get_position() for ax in axes]
    
        x0 = min(b.x0 for b in boxes) - pad
        y0 = min(b.y0 for b in boxes) - pad
        x1 = max(b.x1 for b in boxes) + pad
        y1 = max(b.y1 for b in boxes) + pad
    
        rect = patches.Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            transform=fig.transFigure,
            fill=False,
            linewidth=1.2
        )
    
        fig.patches.append(rect)
    '''
    
    def quality_label(score):
        if score >= 80: return "Excellent"
        elif score >= 60: return "Good"
        elif score >= 40: return "Moderate"
        elif score >= 20: return "Low"
        else: return "Poor"
    
    def quality_color(score):
        if score >= 80: return "green"
        elif score >= 60: return "yellowgreen"
        elif score >= 40: return "orange"
        elif score >= 20: return "orangered"
        else: return "red"
    
            
    def put_border(ax, lw=0.8):
        for s in ax.spines.values():
            s.set_visible(True)
            s.set_linewidth(lw)
            s.set_edgecolor("black")
    
    def wrap_text(text, width=80):
        return "\n".join(textwrap.wrap(text, width))
    
    # -------------------------
    # CRS-SAFE POLYGON REPROJECT
    # -------------------------
    def get_polygon_in_crs(poly, orig_crs, target_crs):
        return gpd.GeoSeries([poly], crs=orig_crs).to_crs(target_crs).iloc[0]
    
    
    # -------------------------
    # S2 (full bounding box)
    # -------------------------
    def extract_s2_rgb_bbox(poly4326, s2_path):
        with rasterio.open(s2_path) as src:
    
            poly_s2 = get_polygon_in_crs(poly4326, "EPSG:4326", src.crs)
            minx, miny, maxx, maxy = poly_s2.bounds
    
            window = rasterio.windows.from_bounds(minx, miny, maxx, maxy, src.transform)
            arr = src.read(window=window).astype(float)
    
            arr -= arr.min()
            if arr.max() > 0:
                arr /= arr.max()
    
            affine = src.window_transform(window)
            return arr, affine, src.crs
    
    
    # -------------------------
    # NDVI polygon stats
    # -------------------------
    
    def extract_ndvi_vals(poly4326, ndvi_path):
    
        # Ensure we always work with geometry, not Series
        if isinstance(poly4326, gpd.GeoSeries):
            poly4326 = poly4326.iloc[0]
    
        with rasterio.open(ndvi_path) as src:
            poly_ndvi = get_polygon_in_crs(poly4326, "EPSG:4326", src.crs)
    
            arr, _ = rio_mask(src, [mapping(poly_ndvi)], crop=True)
    
            band = arr[0].astype(float)
    
            if np.ma.isMaskedArray(band):
                vals = band.compressed()
            else:
                vals = band.ravel()
    
            vals = vals[np.isfinite(vals)]
            vals = vals[vals > -0.1]
            return vals
    
    
    # -------------------------
    # NDVI full bounding box image
    # -------------------------
    def extract_ndvi_bbox(poly4326, ndvi_path):
        with rasterio.open(ndvi_path) as src:
    
            poly_ndvi = get_polygon_in_crs(poly4326, "EPSG:4326", src.crs)
            minx, miny, maxx, maxy = poly_ndvi.bounds
    
            window = rasterio.windows.from_bounds(minx, miny, maxx, maxy, src.transform)
            nd = src.read(1, window=window).astype(float)
    
            nd -= nd.min()
            if nd.max() > 0:
                nd /= nd.max()
    
            affine = src.window_transform(window)
            return nd, affine, src.crs
    
    
    # -------------------------
    # Pixel mapping FIXED
    # -------------------------
    def geom_to_crop_pixels(geom4326, affine, raster_crs):
        g = gpd.GeoSeries([geom4326], crs="EPSG:4326").to_crs(raster_crs).iloc[0]
    
        inv = ~affine
        segs = []
    
        if g.geom_type == "Polygon":
            rings = [g.exterior]
        elif g.geom_type == "MultiPolygon":
            rings = [part.exterior for part in g.geoms]
        else:
            return []
    
        for ring in rings:
            pts = []
            for x, y in ring.coords:
                col, row = inv * (x, y)
                pts.append((col, row))
            segs.append(pts)
    
        return segs
    
    
    # -------------------------
    # RGB & NDVI histograms
    # -------------------------
    def plot_rgb_hist(ax, rgb):
        colors = ["red", "green", "blue"]
        for i, col in enumerate(colors):
            ax.hist(rgb[i].ravel(), bins=256, color=col, alpha=0.5)
        ax.set_title("RGB Histogram")
    
    
    def plot_ndvi_hist(ax, vals):
        vals = vals[np.isfinite(vals)]
        ax.hist(vals, bins=40, color="green", alpha=0.7)
        ax.set_title("NDVI Distribution")
        ax.set_xlabel("NDVI")
        ax.set_ylabel("Count")
    
    
    
    # =====================================================
    # PDF GENERATION — SCIENTIFIC REPORT STYLE
    # =====================================================
    
    with PdfPages(output_pdf) as pdf:
        
        # ============================================================
        # FIRST PAGE — EXPLANATION / LEGEND
        # ============================================================
        fig = plt.figure(figsize=(8.27, 11.69), constrained_layout=True)
        
        gs = gridspec.GridSpec(
            4, 2,
            figure=fig,
            height_ratios=[1.4, 2.6, 2.6, 2.2]
        )
        
        # -------------------------
        # HEADER (TITLE + CLASS INFO)
        # -------------------------
        ax_header = fig.add_subplot(gs[0, :])
        ax_header.axis("off")
        
        ax_header.text(
            0.01, 0.9,
            "CLC Land Cover Sample Quality Report",
            fontsize=16,
            fontweight="bold",
            va="top"
        )
        ax_header.text(
            0.01, 0.75,
            "This page explains the contents of each panel in the following pages.",
            fontsize=11,
            style="italic",
            va="top"
        )
        
        # -------------------------
        # LEFT PANEL — Reference Image Explanation
        # -------------------------
        ax_img = fig.add_subplot(gs[1, 0])
        ax_img.axis("off")
        
        ax_img.text(
            0.02, 0.95,
            "Reference Image",
            fontsize=11,
            fontweight="bold",
            va="top"
        )
        
        ax_img.text(
            0.02, 0.80,
            "Example photograph representing\n the land cover class.",
            fontsize=10,
            va="top",
            wrap=True
        )
        
        put_border(ax_img)
        
        # -------------------------
        # RIGHT PANEL — Statistics Explanation
        # -------------------------
        ax_stats = fig.add_subplot(gs[1, 1])
        ax_stats.axis("off")
        
        ax_stats.text(
            0.02, 0.95,
            "Sample statistics",
            fontsize=11,
            fontweight="bold",
            va="top"
        )
        
        stats_expl = (
            "• OBJECTID: Polygon identifier\n"
            "• NDVI mean, std, range: Vegetation index statistics\n"
            "• Interior margin: Distance from polygon boundary\n"
            "• Quality score: Indicates how representative the sample is\n"
            "  of the land cover class.\n"
            "    High → homogeneous, reliable sample\n"
            "    Medium → moderate variability\n"
            "    Low → heterogeneous or less reliable\n"
            "  (color-coded from red = low to green = high)"
        )
        
        ax_stats.text(
            0.02, 0.80,
            stats_expl,
            fontsize=10,
            va="top"
        )
        
        put_border(ax_stats)
        
        # -------------------------
        # MAP PANELS — Sentinel-2 RGB + NDVI Explanation
        # -------------------------
        gs_maps = gridspec.GridSpecFromSubplotSpec(
            1, 2, subplot_spec=gs[2, :], wspace=0.15
        )
        
        # RGB
        ax_rgb = fig.add_subplot(gs_maps[0, 0])
        ax_rgb.axis("off")
        
        ax_rgb.text(
            0.02, 0.95,
            "Sentinel-2 RGB Image",
            fontsize=11,
            fontweight="bold",
            va="top"
        )
        
        ax_rgb.text(
            0.02, 0.80,
            "True color satellite image.\n"
            "Red = CLC polygon\n"
            "Yellow = selected sample",
            fontsize=10,
            va="top"
        )
        
        put_border(ax_rgb)
        
        # NDVI
        ax_ndvi = fig.add_subplot(gs_maps[0, 1])
        ax_ndvi.axis("off")
        
        ax_ndvi.text(
            0.02, 0.95,
            "NDVI Map",
            fontsize=11,
            fontweight="bold",
            va="top"
        )
        
        ax_ndvi.text(
            0.02, 0.80,
            "Vegetation index visualization.\n"
            "Higher green values = denser vegetation.",
            fontsize=10,
            va="top"
        )
        
        put_border(ax_ndvi)
        
        # -------------------------
        # HISTOGRAMS Explanation
        # -------------------------
        ax_rgb_h = fig.add_subplot(gs[3, 0])
        ax_rgb_h.axis("off")
        
        ax_rgb_h.text(
            0.02, 0.95,
            "RGB Histogram",
            fontsize=11,
            fontweight="bold",
            va="top"
        )
        
        ax_rgb_h.text(
            0.02, 0.80,
            "Distribution of pixel intensities for each band.",
            fontsize=10,
            va="top"
        )
        
        put_border(ax_rgb_h)
        
        ax_ndvi_h = fig.add_subplot(gs[3, 1])
        ax_ndvi_h.axis("off")
        
        ax_ndvi_h.text(
            0.02, 0.95,
            "NDVI Histogram",
            fontsize=11,
            fontweight="bold",
            va="top"
        )
        
        ax_ndvi_h.text(
            0.02, 0.80,
            "Distribution of vegetation index values.",
            fontsize=10,
            va="top"
        )
        
        put_border(ax_ndvi_h)
        
        # ============================================================
        # SAVE FIRST PAGE
        # ============================================================
        pdf.savefig(fig)
        plt.close(fig)
    
        # ============================================================
        # FOLLOWING PAGES — ACTUAL DATA
        # ============================================================
        
        for cl in sorted(final["clc_l3"].unique()):
            print(f"Building page for class {cl}...")
    
            meta = clc_metadata[int(cl)]
    
            # -------------------------
            # SELECT BEST SAMPLE
            # -------------------------
            g = final[final["clc_l3"] == cl].sort_values("quality_score", ascending=False)
            best = g.iloc[0]
    
            object_id = best["OBJECTID"]
            sample3035 = best.geometry
    
            gcl = gpd.read_file(os.path.join(root_dir, "split_outputs", f"CLC_Class_{cl}.gpkg"))
            parent_poly3035 = gcl.loc[gcl["OBJECTID"] == object_id, "geometry"].iloc[0]
    
            parent4326 = get_polygon_in_crs(parent_poly3035, "EPSG:3035", "EPSG:4326")
            sample4326 = get_polygon_in_crs(sample3035, "EPSG:3035", "EPSG:4326")
    
            # -------------------------
            # DATA EXTRACTION
            # -------------------------
            rgb, affine_s2, s2_crs = extract_s2_rgb_bbox(parent4326, s2_path)
            ndvi_vals = extract_ndvi_vals(parent4326, raster_ndvi_path)
            ndvi_img, affine_ndvi, ndvi_crs = extract_ndvi_bbox(parent4326, raster_ndvi_path)
    
            # -------------------------
            # LOAD EXAMPLE IMAGE
            # -------------------------
            example_img = None
            folder = meta["example_folder"]
    
            if folder:
                fp = os.path.join(root_dir, "example_images", folder)
                if os.path.exists(fp):
                    jpgs = [f for f in os.listdir(fp) if f.lower().endswith(".jpg")]
                    if jpgs:
                        import matplotlib.image as mpimg
                        example_img = mpimg.imread(os.path.join(fp, jpgs[0]))
    
            # ============================================================
            # FIGURE LAYOUT
            # ============================================================
            fig = plt.figure(figsize=(8.27, 11.69), constrained_layout=True)
    
            gs = gridspec.GridSpec(
                4, 2,
                figure=fig,
                height_ratios=[1.4, 2.6, 2.6, 2.2]
            )
            
            # ============================================================
            # HEADER (TITLE + CLASS INFO)
            # ============================================================
            ax_header = fig.add_subplot(gs[0, :])
            ax_header.axis("off")
            
            # -------------------------
            # TEXT CONTENT
            # -------------------------
            title = f"CLC Class {cl} — {meta['level3']}"
            title_wrapped = wrap_text(title, width=60)
            
            subtitle = f"{meta['level1']}  |  {meta['level2']}"
            subtitle_wrapped = wrap_text(subtitle, width=60)
            
            desc_wrapped = wrap_text(meta["description"], width=110)
            
            # -------------------------
            # DRAW + MEASURE
            # -------------------------
            y = 0.95
            
            # --- TITLE ---
            t1 = ax_header.text(
                0.01, y,
                title_wrapped,
                fontsize=14,
                fontweight="bold",
                va="top"
            )
            
            fig.canvas.draw()  # ⬅️ IMPORTANT: compute size
            bbox1 = t1.get_window_extent(renderer=fig.canvas.get_renderer())
            
            # Convert height from pixels → axis coords
            inv = ax_header.transAxes.inverted()
            bbox1_axes = bbox1.transformed(inv)
            y = bbox1_axes.y0 - 0.1   # small gap
            
            # --- SUBTITLE ---
            t2 = ax_header.text(
                0.01, y,
                subtitle_wrapped,
                fontsize=11,
                style="italic",
                va="top"
            )
            
            fig.canvas.draw()
            bbox2 = t2.get_window_extent(renderer=fig.canvas.get_renderer())
            bbox2_axes = bbox2.transformed(inv)
            y = bbox2_axes.y0 - 0.2
            
            # --- DESCRIPTION ---
            ax_header.text(
                0.01, y,
                desc_wrapped,
                fontsize=10,
                va="top"
            )
    
            # ============================================================
            # LEFT: EXAMPLE IMAGE
            # ============================================================
    
            ax_img = fig.add_subplot(gs[1, 0])
            #ax_img.set_title("Reference example", fontsize=11)
    
            if example_img is not None:
                ax_img.imshow(example_img)
            else:
                ax_img.text(0.5, 0.5, "No image available", ha="center")
    
            ax_img.set_aspect('equal')   # keeps proportions
            ax_img.set_anchor('C')       # center image
    
            ax_img.axis("off")
            put_border(ax_img)
                
            # ============================================================
            # RIGHT: STATISTICS PANEL
            # ============================================================
            ax_stats = fig.add_subplot(gs[1, 1])
            ax_stats.axis("off")
            
            # --- Title ---
            ax_stats.text(
                0.02, 0.98,
                "Sample statistics",
                fontsize=11,
                fontweight="bold",
                va="top"
            )
            
            # -------------------------
            # PREP DATA
            # -------------------------
            q = best["quality_score"]
            label = quality_label(q)
            color = quality_color(q)
            
            # --- Clean strings (NO indentation!) ---
            stats_top = f"""
            OBJECTID: {object_id}
            
            NDVI mean: {best['ndvi_mean']:.3f}
            NDVI std: {best['ndvi_std']:.3f}
            NDVI range: {best['ndvi_range']:.3f}
            """.strip()
            
            quality_text = textwrap.dedent(f"""
            Quality score: {q:.1f}
            Quality class: {label}
            """).strip()
            
            stats_bottom = f"""
            Interior margin: {best['interior_margin_m']:.1f} m
            
            Legend:
            Red = CLC polygon
            Yellow = sample
            """.strip()
            
            # -------------------------
            # DYNAMIC LAYOUT
            # -------------------------
            y = 0.88
            
            # --- TOP BLOCK ---
            t_top = ax_stats.text(
                0.02, y,
                stats_top,
                fontsize=10,
                va="top"
            )
            
            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            inv = ax_stats.transAxes.inverted()
            
            bbox_top = t_top.get_window_extent(renderer=renderer).transformed(inv)
            y = bbox_top.y0 - 0.05
            
            # --- QUALITY BLOCK ---
            t_q = ax_stats.text(
                0.02, y,
                quality_text,
                fontsize=10,
                va="top",
                color=color
            )
            
            fig.canvas.draw()
            bbox_q = t_q.get_window_extent(renderer=renderer).transformed(inv)
            y = bbox_q.y0 - 0.05
            
            # --- BOTTOM BLOCK ---
            ax_stats.text(
                0.02, y,
                stats_bottom,
                fontsize=10,
                va="top"
            )
            
            put_border(ax_stats)
    
            # ============================================================
            # MAP PANELS (FIGURE STYLE)
            # ============================================================
            gs_maps = gridspec.GridSpecFromSubplotSpec(
                1, 2, subplot_spec=gs[2, :], wspace=0.15
            )
    
            # ---- RGB PANEL ----
            ax_rgb = fig.add_subplot(gs_maps[0, 0])
            ax_rgb.imshow(np.moveaxis(rgb, 0, -1))
            ax_rgb.set_title("Sentinel-2 RGB", fontsize=11)
            ax_rgb.axis("off")
    
            parent_s2 = gpd.GeoSeries([parent4326], crs="EPSG:4326").to_crs(s2_crs).iloc[0]
            sample_s2 = gpd.GeoSeries([sample4326], crs="EPSG:4326").to_crs(s2_crs).iloc[0]
    
            for seg in geom_to_crop_pixels(parent_s2, affine_s2, s2_crs):
                xs, ys = zip(*seg)
                ax_rgb.plot(xs, ys, "-r", lw=1.5)
    
            for seg in geom_to_crop_pixels(sample_s2, affine_s2, s2_crs):
                xs, ys = zip(*seg)
                ax_rgb.plot(xs, ys, "-y", lw=1.5)
    
            put_border(ax_rgb)
    
            # ---- NDVI PANEL ----
            ax_ndvi = fig.add_subplot(gs_maps[0, 1])
            ax_ndvi.imshow(ndvi_img, cmap="RdYlGn")
            ax_ndvi.set_title("NDVI", fontsize=11)
            ax_ndvi.axis("off")
    
            parent_ndvi = gpd.GeoSeries([parent4326], crs="EPSG:4326").to_crs(ndvi_crs).iloc[0]
            sample_ndvi = gpd.GeoSeries([sample4326], crs="EPSG:4326").to_crs(ndvi_crs).iloc[0]
    
            for seg in geom_to_crop_pixels(parent_ndvi, affine_ndvi, ndvi_crs):
                xs, ys = zip(*seg)
                ax_ndvi.plot(xs, ys, "-r", lw=1.5)
    
            for seg in geom_to_crop_pixels(sample_ndvi, affine_ndvi, ndvi_crs):
                xs, ys = zip(*seg)
                ax_ndvi.plot(xs, ys, "-y", lw=1.5)
    
            put_border(ax_ndvi)
    
            # ============================================================
            # HISTOGRAMS
            # ============================================================
            ax_rgb_h = fig.add_subplot(gs[3, 0])
            plot_rgb_hist(ax_rgb_h, rgb)
            ax_rgb_h.set_title("RGB distribution", fontsize=11)
            put_border(ax_rgb_h)
    
            ax_ndvi_h = fig.add_subplot(gs[3, 1])
            if len(ndvi_vals) > 0:
                plot_ndvi_hist(ax_ndvi_h, ndvi_vals)
            else:
                ax_ndvi_h.text(0.3, 0.5, "No NDVI data")
    
            ax_ndvi_h.set_title("NDVI distribution", fontsize=11)
            put_border(ax_ndvi_h)
    
            # ============================================================
            # SAVE PAGE
            # ============================================================
            pdf.savefig(fig)
            plt.close(fig)
    
    print("Summary PDF created:", output_pdf)


with StepTimer("Exporting pdf per class..."):
    # ============================================================
    # NEW FEATURE — ONE PDF PER CLASS (60 samples)
    # ============================================================
    
    gcl_cache = {
        cl: gpd.read_file(os.path.join(root_dir, "split_outputs", f"CLC_Class_{cl}.gpkg"))
        for cl in final["clc_l3"].unique()
    }
    
    for cl in sorted(final["clc_l3"].unique()):
        df_cl = final[
            final["clc_l3"] == cl
        ].sort_values("quality_score", ascending=False)
    
        output_pdf = f"{export_dir}/pdf/CLC_{cl}_samples.pdf"
    
        with PdfPages(output_pdf) as pdf:
    
            # ====================================================
            # OPTIONAL: FIRST PAGE (CLASS INTRO)
            # ====================================================
            fig = plt.figure(figsize=(8.27, 11.69))
            ax = fig.add_subplot(111)
            ax.axis("off")
    
            meta = clc_metadata.get(int(cl), {})
            class_name = meta.get("level3", "Unknown class")
            
            ax.text(
                0.5, 0.7,
                f"CLC Class {cl}",
                ha="center", va="center",
                fontsize=16, fontweight="bold"
            )
            
            ax.text(
                0.5, 0.45,
                class_name,
                ha="center", va="center",
                fontsize=12
            )
            
            ax.text(
                0.5, 0.2,
                "Top 60 samples ranked by quality score",
                ha="center", va="center",
                fontsize=11
            )
    
            pdf.savefig(fig)
            plt.close(fig)
    
            # ====================================================
            # ONE PAGE PER SAMPLE (reuse your layout)
            # ====================================================
            for i, (_, row) in enumerate(df_cl.iterrows()):
    
                fig = plt.figure(figsize=(8.27, 11.69), constrained_layout=True)
    
                gs = gridspec.GridSpec(
                    4, 2,
                    figure=fig,
                    height_ratios=[1.4, 2.6, 2.6, 2.2]
                )
    
                # -------------------------
                # HEADER
                # -------------------------
                ax_header = fig.add_subplot(gs[0, :])
                ax_header.axis("off")
    
                ax_header.text(
                    0.01, 0.9,
                    f"CLC Class {cl} — OBJECTID {row['OBJECTID']}",
                    fontsize=14,
                    fontweight="bold",
                    va="top"
                )
    
                ax_header.text(
                    0.01, 0.65,
                    f"Quality score: {row['quality_score']:.2f}",
                    fontsize=11,
                    style="italic"
                )
    
                # page number
                ax_header.text(
                    0.98, 0.9,
                    f"{i+1}/{len(df_cl)}",
                    ha="right",
                    fontsize=10
                )
    
                # -------------------------
                # LEFT: IMAGE
                # -------------------------
                ax_img = fig.add_subplot(gs[1, 0])
                ax_img.axis("off")
    
                # 👉 reuse your existing image plotting
                # plot_reference_image(ax_img, row)
    
                put_border(ax_img)
    
                # -------------------------
                # RIGHT: STATS
                # -------------------------
                ax_stats = fig.add_subplot(gs[1, 1])
                ax_stats.axis("off")
    
                ax_stats.text(
                    0.02, 0.95,
                    "Sample statistics",
                    fontsize=11,
                    fontweight="bold",
                    va="top"
                )
    
                stats_text = f"""
    OBJECTID: {row['OBJECTID']}
    
    NDVI mean: {row['ndvi_mean']:.3f}
    NDVI std: {row['ndvi_std']:.3f}
    NDVI range: {row['ndvi_range']:.3f}
    
    Interior margin: {row['interior_margin_m']:.1f} m
    """
    
                ax_stats.text(
                    0.02, 0.80,
                    stats_text,
                    fontsize=10,
                    va="top"
                )
    
                put_border(ax_stats)
    
                # -------------------------
                # MAPS (reuse your functions)
                # -------------------------
                gs_maps = gridspec.GridSpecFromSubplotSpec(
                    1, 2, subplot_spec=gs[2, :], wspace=0.15
                )
    
                ax_rgb = fig.add_subplot(gs_maps[0, 0])
                ax_rgb.axis("off")
                # plot_rgb(ax_rgb, row)
                put_border(ax_rgb)
    
                ax_ndvi = fig.add_subplot(gs_maps[0, 1])
                ax_ndvi.axis("off")
                # plot_ndvi(ax_ndvi, row)
                put_border(ax_ndvi)
    
                # -------------------------
                # HISTOGRAMS
                # -------------------------
                # -------------------------
                # DATA EXTRACTION (same as summary)
                # -------------------------
                object_id = row["OBJECTID"]
                sample3035 = row.geometry
                
                #gcl = gpd.read_file(os.path.join(root_dir, "split_outputs", f"CLC_Class_{cl}.gpkg"))
                gcl = gcl_cache[cl]
                parent_poly3035 = gcl.loc[gcl["OBJECTID"] == object_id, "geometry"].iloc[0]
                
                parent4326 = get_polygon_in_crs(parent_poly3035, "EPSG:3035", "EPSG:4326")
                sample4326 = get_polygon_in_crs(sample3035, "EPSG:3035", "EPSG:4326")
                
                rgb, affine_s2, s2_crs = extract_s2_rgb_bbox(parent4326, s2_path)
                ndvi_vals = extract_ndvi_vals(parent4326, raster_ndvi_path)
                ndvi_img, affine_ndvi, ndvi_crs = extract_ndvi_bbox(parent4326, raster_ndvi_path)
                
                ax_rgb = fig.add_subplot(gs_maps[0, 0])
                ax_rgb.imshow(np.moveaxis(rgb, 0, -1))
                ax_rgb.set_title("Sentinel-2 RGB", fontsize=11)
                ax_rgb.axis("off")
                
                parent_s2 = gpd.GeoSeries([parent4326], crs="EPSG:4326").to_crs(s2_crs).iloc[0]
                sample_s2 = gpd.GeoSeries([sample4326], crs="EPSG:4326").to_crs(s2_crs).iloc[0]
                
                for seg in geom_to_crop_pixels(parent_s2, affine_s2, s2_crs):
                    xs, ys = zip(*seg)
                    ax_rgb.plot(xs, ys, "-r", lw=1.5)
                
                for seg in geom_to_crop_pixels(sample_s2, affine_s2, s2_crs):
                    xs, ys = zip(*seg)
                    ax_rgb.plot(xs, ys, "-y", lw=1.5)
                
                put_border(ax_rgb)
                
                ax_ndvi = fig.add_subplot(gs_maps[0, 1])
                ax_ndvi.imshow(ndvi_img, cmap="RdYlGn")
                ax_ndvi.set_title("NDVI", fontsize=11)
                ax_ndvi.axis("off")
                
                parent_ndvi = gpd.GeoSeries([parent4326], crs="EPSG:4326").to_crs(ndvi_crs).iloc[0]
                sample_ndvi = gpd.GeoSeries([sample4326], crs="EPSG:4326").to_crs(ndvi_crs).iloc[0]
                
                for seg in geom_to_crop_pixels(parent_ndvi, affine_ndvi, ndvi_crs):
                    xs, ys = zip(*seg)
                    ax_ndvi.plot(xs, ys, "-r", lw=1.5)
                
                for seg in geom_to_crop_pixels(sample_ndvi, affine_ndvi, ndvi_crs):
                    xs, ys = zip(*seg)
                    ax_ndvi.plot(xs, ys, "-y", lw=1.5)
                
                put_border(ax_ndvi)
    
                ax_ndvi = fig.add_subplot(gs_maps[0, 1])
                ax_ndvi.imshow(ndvi_img, cmap="RdYlGn")
                ax_ndvi.set_title("NDVI", fontsize=11)
                ax_ndvi.axis("off")
                
                parent_ndvi = gpd.GeoSeries([parent4326], crs="EPSG:4326").to_crs(ndvi_crs).iloc[0]
                sample_ndvi = gpd.GeoSeries([sample4326], crs="EPSG:4326").to_crs(ndvi_crs).iloc[0]
                
                for seg in geom_to_crop_pixels(parent_ndvi, affine_ndvi, ndvi_crs):
                    xs, ys = zip(*seg)
                    ax_ndvi.plot(xs, ys, "-r", lw=1.5)
                
                for seg in geom_to_crop_pixels(sample_ndvi, affine_ndvi, ndvi_crs):
                    xs, ys = zip(*seg)
                    ax_ndvi.plot(xs, ys, "-y", lw=1.5)
                
                put_border(ax_ndvi)
                
                ax_rgb_h = fig.add_subplot(gs[3, 0])
                plot_rgb_hist(ax_rgb_h, rgb)
                ax_rgb_h.set_title("RGB distribution", fontsize=11)
                put_border(ax_rgb_h)
                
                ax_ndvi_h = fig.add_subplot(gs[3, 1])
                if len(ndvi_vals) > 0:
                    plot_ndvi_hist(ax_ndvi_h, ndvi_vals)
                else:
                    ax_ndvi_h.text(0.3, 0.5, "No NDVI data")
                
                ax_ndvi_h.set_title("NDVI distribution", fontsize=11)
                put_border(ax_ndvi_h)
                # -------------------------
                # SAVE
                # -------------------------
                pdf.savefig(fig)
                plt.close(fig)
    
    print("Per class PDF created:", output_pdf)